# weights

> Load a wav2vec 2.0 state dict onto the MAX module.
> 
> Checkpoint keys follow the Hugging Face `Wav2Vec2ForCTC` names. Some
> entries are not parameters of the graph: the SpecAugment mask embedding
> and BatchNorm's step counter. The position convolution is stored with
> weight norm, as a gain and a direction. `fold_weight_norm` turns that pair
> into one ordinary weight.


In [ ]:
#| default_exp weights


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from pathlib import Path

import numpy as np
from max.dtype import DType
from max.experimental.tensor import Tensor


In [ ]:
#| export
_NUMPY = {
    DType.float32: np.float32,
    DType.float64: np.float64,
    DType.int64: np.int64,
    DType.int32: np.int32,
}


In [ ]:
#| export
def usable(key: str) -> bool:
    """True when `key` is a tensor the MAX module stores."""
    if key.endswith("masked_spec_embed"):
        return False
    if key.endswith("num_batches_tracked"):
        return False
    if key.endswith("embeddings.position_ids"):
        return False
    return True


Older checkpoints name the weight-norm pair `weight_g` and `weight_v`. Newer PyTorch saves it as `parametrizations.weight.original0` and `original1`. Both are the gain `g` and the direction `v`. The convolution weight is `v * g / ||v||`, where the norm is taken over every axis except the kernel axis (`dim=2`).


In [ ]:
#| export
_PAIRS = (
    (".weight_g", ".weight_v"),
    (".parametrizations.weight.original0", ".parametrizations.weight.original1"),
)


In [ ]:
#| export
def fold_weight_norm(arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Replace each weight-norm `(g, v)` pair with the single weight it defines."""
    folded = dict(arrays)
    for gain_suffix, direction_suffix in _PAIRS:
        for key in [key for key in arrays if key.endswith(gain_suffix)]:
            base = key[: -len(gain_suffix)]
            gain = folded.pop(key)
            direction = folded.pop(base + direction_suffix)
            folded[base + ".weight"] = _fold(gain, direction)
    return folded


In [ ]:
#| export
def _fold(gain: np.ndarray, direction: np.ndarray) -> np.ndarray:
    direction = np.asarray(direction, dtype=np.float64)
    axes = tuple(axis for axis in range(direction.ndim) if axis != 2)
    norm = np.sqrt((direction**2).sum(axis=axes, keepdims=True))
    gain = np.asarray(gain, dtype=np.float64).reshape(norm.shape)
    return np.ascontiguousarray(direction * (gain / norm), dtype=np.float32)


In [ ]:
#| export
def load_arrays(path: Path) -> dict[str, np.ndarray]:
    """Read a `.safetensors` or `.npz` checkpoint, dropping unused buffers and folding weight norm."""
    path = Path(path)
    if path.suffix == ".safetensors":
        from safetensors.numpy import load_file

        arrays = load_file(path)
    elif path.suffix == ".npz":
        with np.load(path) as data:
            arrays = {key: data[key] for key in data.files}
    else:
        raise ValueError(f"{path.name} is not a .safetensors or .npz checkpoint")
    return fold_weight_norm({key: value for key, value in arrays.items() if usable(key)})


In [ ]:
#| export
def assign_parameters(module, arrays: dict[str, np.ndarray]) -> None:
    """Copy `arrays` into `module`. Names and shapes must match."""
    wanted = dict(module.parameters)
    missing = [name for name in wanted if name not in arrays]
    extra = [name for name in arrays if name not in wanted]
    if missing or extra:
        sample = (missing or extra)[:6]
        raise KeyError(
            f"checkpoint does not match the module "
            f"({len(missing)} missing, {len(extra)} extra, first {sample})"
        )
    cast = {name: _fit(arrays[name], tensor) for name, tensor in wanted.items()}
    module.load_state_dict(cast, strict=True)


In [ ]:
#| export
def _fit(array: np.ndarray, tensor: Tensor) -> np.ndarray:
    """Cast to the parameter dtype and reshape arrays whose size matches."""
    expected = tuple(int(dim) for dim in tensor.shape)
    array = np.ascontiguousarray(array, dtype=_NUMPY[tensor.dtype])
    if array.shape != expected:
        if array.size != int(np.prod(expected, dtype=int, initial=1)):
            raise ValueError(f"shape {array.shape} cannot fill {expected}")
        array = np.ascontiguousarray(array.reshape(expected))
    return array


Folding keeps the kernel axis last and leaves everything else alone. With a gain of one, each filter slice along that axis comes out unit length.


In [ ]:
v = np.random.default_rng(0).standard_normal((6, 2, 4)).astype(np.float32)
g = np.ones((1, 1, 4), dtype=np.float32)
w = fold_weight_norm({"conv.weight_g": g, "conv.weight_v": v, "conv.bias": np.zeros(6, np.float32)})
assert sorted(w) == ["conv.bias", "conv.weight"]
assert np.allclose(np.sqrt((w["conv.weight"] ** 2).sum(axis=(0, 1))), 1.0, atol=1e-5)
